# Jalon 2 — Nettoyage des données et calcul des KPI
**Stage PFA — Tableau de bord décisionnel EMC Helpline**
Soukaina MLLOUK — Encadrante : Mme Rachida Margdane

Objectif de ce notebook :
1. Charger et explorer le dataset brut
2. Nettoyer et standardiser les colonnes
3. Calculer les 8 KPI validés par l'encadrante
4. Produire un premier graphique Plotly par KPI
5. Effectuer des contrôles qualité finaux
6. Vérifier manuellement quelques calculs

> ⚠️ KPI 6 (« pour vous / pour quelqu'un ») a été retiré .
> Les colonnes `titulaire` et `emetteur` sont donc supprimées : elles ne servent plus à aucun KPI retenu.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

## 1. Chargement des données



In [2]:
df = pd.read_excel(r"C:\Users\Ce Pc\Desktop\cmrpi\signalements.xlsx")

print("Dimensions :", df.shape)
df.head()

Dimensions : (138, 12)


,id,titulaire,emetteur,cyberharcelementType,plateforme,accompagnement,date,genre,age,typeAccompagnement,langue,anonymat
0,979,Oui,public,Publication de photos intimes ou personnelles,Facebook,Non,2025-01-01,Masculin,Plus de 26 ans,Suppression,fr,Oui
1,980,Non,public,Publication de photos intimes ou personnelles,Instagram,Non,2025-01-02,Féminin,Âges de 18 à 25 ans,Suppression,fr,Oui
2,981,Oui,public,Publication de photos intimes ou personnelles,Facebook,Oui,2025-01-03,Féminin,Âges de 18 à 25 ans,Juridique;Psychique;Suppression,ar,Non
3,982,Oui,public,Autres,Facebook,Non,2025-01-04,Féminin,Plus de 26 ans,Suppression,fr,Oui
4,983,Non,public,Autres,Facebook,Non,2025-01-05,Féminin,Âges de 18 à 25 ans,Suppression,fr,Oui


## 2. Exploration initiale

On revérifie les types de colonnes et les valeurs manquantes avant de nettoyer.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    138 non-null    int64         
 1   titulaire             128 non-null    str           
 2   emetteur              128 non-null    str           
 3   cyberharcelementType  138 non-null    str           
 4   plateforme            138 non-null    str           
 5   accompagnement        138 non-null    str           
 6   date                  138 non-null    datetime64[us]
 7   genre                 133 non-null    str           
 8   age                   129 non-null    str           
 9   typeAccompagnement    138 non-null    str           
 10  langue                138 non-null    str           
 11  anonymat              138 non-null    str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 13.1 KB


In [4]:
for col in df.columns:
    print(f"--- {col} ---")
    print(df[col].unique())
    print(f"Valeurs manquantes : {df[col].isnull().sum()}\n")

--- id ---
[ 979  980  981  982  983  984  985  986  987  988  989  990  991  992
  993  994  995  996  997  998  999 1000 1001 1002 1003 1004 1005 1006
 1007 1008 1009 1010 1011 1012 1013 1014 1015 1016 1017 1018 1019 1020
 1021 1022 1023 1024 1025 1026 1027 1028 1029 1030 1031 1032 1033 1034
 1035 1036 1037 1038 1039 1040 1041 1042 1351 1352 1353 1354 1355 1356
 1357 1358 1359 1360 1361 1362 1363 1364 1365 1366 1367 1368 1369 1370
 1371 1372 1373 1374 1375 1376 1377 1378 1379 1380 1381 1382 1383 1384
 1385 1386 1387 1388 1389 1390 1391 1392 1393 1394 1395 1396 1397 1398
 1399 1400 1401 1402 1403 1404 1405 1406 1407 1408 1409 1410 1411 1412
 1413 1414 1668 1669 1670 1671 1672 1673 1674 1675 1676 1677]
Valeurs manquantes : 0

--- titulaire ---
<StringArray>
['Oui', 'Non', nan]
Length: 3, dtype: str
Valeurs manquantes : 10

--- emetteur ---
<StringArray>
['public', nan]
Length: 2, dtype: str
Valeurs manquantes : 10

--- cyberharcelementType ---
<StringArray>
[       'Publication de phot

In [5]:
# On garde une trace des valeurs manquantes AVANT nettoyage, pour la documentation
nan_avant = {
    "genre": df["genre"].isnull().sum(),
    "age": df["age"].isnull().sum(),
}
nan_avant

{'genre': np.int64(5), 'age': np.int64(9)}

## 3. Nettoyage et standardisation

### 3.1 Suppression des colonnes non utilisées

- `titulaire` : ne servait qu'au KPI 6, retiré par l'encadrante → colonne supprimée.
- `emetteur` : quasi constante ("public"), déjà écartée de l'analyse dans le rapport Jalon 1 → colonne supprimée.

On ne cherche pas à remplir leurs valeurs manquantes puisqu'elles sont retirées.

In [6]:
df = df.drop(columns=["titulaire", "emetteur"])
df.columns

Index(['id', 'cyberharcelementType', 'plateforme', 'accompagnement', 'date',
       'genre', 'age', 'typeAccompagnement', 'langue', 'anonymat'],
      dtype='str')

### 3.2 Nettoyage de la date

La colonne `date` est déjà au format datetime, mais on s'assure qu'il n'y a pas d'heure
résiduelle (on garde uniquement la partie date).

In [7]:
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["annee"] = df["date"].dt.year
df["mois"] = df["date"].dt.month
df[["date", "annee", "mois"]].head()

,date,annee,mois
0,2025-01-01,2025,1
1,2025-01-02,2025,1
2,2025-01-03,2025,1
3,2025-01-04,2025,1
4,2025-01-05,2025,1


In [8]:
print("ID min/max :", df['id'].min(), df['id'].max())
print("Nombre de lignes :", len(df))
print("Période couverte :", df['date'].min(), "→", df['date'].max())

ID min/max : 979 1677
Nombre de lignes : 138
Période couverte : 2025-01-01 00:00:00 → 2025-12-17 00:00:00


### 3.3 Standardisation de `accompagnement`

Valeurs actuelles : `Non`, `Oui`, `oui`, `OUI` → il faut uniformiser la casse.



In [9]:
df["accompagnement"] = df["accompagnement"].str.strip().str.capitalize()
df["accompagnement"].unique()

<StringArray>
['Non', 'Oui']
Length: 2, dtype: str

### 3.4 Standardisation de `langue`

Valeurs actuelles : `fr`, `ar`, `FR` → on uniformise en minuscules.

In [10]:
df["langue"] = df["langue"].str.strip().str.lower()
df["langue"].unique()

<StringArray>
['fr', 'ar']
Length: 2, dtype: str

### 3.5 Nettoyage de `cyberharcelementType`

Certaines valeurs ont des espaces parasites en début/fin (ex. `" Diffamation"`,
`"Escroquerie "`). On nettoie avec `.str.strip()`.

In [11]:
df["cyberharcelementType"] = df["cyberharcelementType"].str.strip()
df["cyberharcelementType"].unique()

<StringArray>
[       'Publication de photos intimes ou personnelles',
                                               'Autres',
                                      'Propos de haine',
 'Menace de publier des photos intimes ou personnelles',
                                          'Diffamation',
                    'Propos raciste ou discriminatoire',
                                'Usurpation d’identité',
                                          'Escroquerie']
Length: 8, dtype: str

### 3.6 Valeurs manquantes de `genre` et `age`

Deux options possibles pour des données catégorielles :
1. **Supprimer les lignes** avec valeur manquante (perte d'information, ok si peu de lignes concernées)
2. **Créer une catégorie "Non renseigné"** (on garde toutes les lignes, visible dans les graphiques)

On choisit l'option 2 ici : on garde toutes les lignes, aucune observation n'est supprimée.

In [12]:
df["genre"] = df["genre"].fillna("Non renseigné")
df["age"] = df["age"].fillna("Non renseigné")

print(df["genre"].value_counts())
print()
print(df["age"].value_counts())

genre
Féminin          79
Masculin         54
Non renseigné     5
Name: count, dtype: int64

age
Âges de 18 à 25 ans    69
Plus de 26 ans         59
Non renseigné           9
Âges de 13 à 17 ans     1
Name: count, dtype: int64


**Documentation du traitement des valeurs manquantes** (avant → après) :

In [13]:
documentation_nan = pd.DataFrame({
    "Colonne": ["genre", "age"],
    "NaN avant": [nan_avant["genre"], nan_avant["age"]],
    "Traitement": ["Non renseigné", "Non renseigné"],
})
documentation_nan

,Colonne,NaN avant,Traitement
0,genre,5,Non renseigné
1,age,9,Non renseigné


### 3.7 Simplification de `age` en tranches lisibles

On raccourcit les libellés pour les graphiques, et on prépare une agrégation
en 3 catégories pour le KPI 3.

Le dataset actuel ne contient aucune ligne "5-12 ans" (cette tranche existe dans le
formulaire mais n'est pas représentée dans cet échantillon) — on ne l'invente pas, on
garde uniquement les catégories réellement présentes : 13-17 ans, 18-25 ans, +26 ans,
Non renseigné.

In [14]:
age_map = {
    "Âges de 13 à 17 ans": "13-17 ans",
    "Âges de 18 à 25 ans": "18-25 ans",
    "Plus de 26 ans": "+26 ans",
    "Non renseigné": "Non renseigné",
}
df["age"] = df["age"].replace(age_map)
df["age"].unique()

<StringArray>
['+26 ans', '18-25 ans', 'Non renseigné', '13-17 ans']
Length: 4, dtype: str

In [15]:
def statut_age(tranche):
    # 3 catégories : les 18-25 ans sont un public à part, plus exposé numériquement
    # que les +26 ans (cf. rapport officiel CMRPI).
    if tranche == "13-17 ans":
        return "Mineur"
    elif tranche == "18-25 ans":
        return "Jeune adulte"
    elif tranche == "+26 ans":
        return "Adulte"
    else:
        return "Non renseigné"

df["statut_age"] = df["age"].apply(statut_age)
df["statut_age"].value_counts()

statut_age
Jeune adulte     69
Adulte           59
Non renseigné     9
Mineur            1
Name: count, dtype: int64

### 3.8 Décomposition de `typeAccompagnement`

Les valeurs comme `"Juridique;Psychique;Suppression"` combinent plusieurs types
d'accompagnement.Je **garde la colonne originale** `typeAccompagnement` (texte, pour
référence) et on **ajoute** trois colonnes binaires (0/1), une par type.

In [16]:
types_possibles = ["Juridique", "Psychique", "Suppression"]

for t in types_possibles:
    df[f"accomp_{t.lower()}"] = df["typeAccompagnement"].apply(
        lambda x: 1 if t in str(x).split(";") else 0
    )

df[["typeAccompagnement", "accomp_juridique", "accomp_psychique", "accomp_suppression"]].head()

,typeAccompagnement,accomp_juridique,accomp_psychique,accomp_suppression
0,Suppression,0,0,1
1,Suppression,0,0,1
2,Juridique;Psychique;Suppression,1,1,1
3,Suppression,0,0,1
4,Suppression,0,0,1


In [17]:
# Vérification des combinaisons réellement présentes dans les données
df["typeAccompagnement"].value_counts()

typeAccompagnement
Suppression                        106
Juridique;Suppression               14
Juridique;Psychique;Suppression     10
Psychique;Suppression                5
Juridique                            3
Name: count, dtype: int64

### 3.9 Vérification finale après nettoyage

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    138 non-null    int64         
 1   cyberharcelementType  138 non-null    str           
 2   plateforme            138 non-null    str           
 3   accompagnement        138 non-null    str           
 4   date                  138 non-null    datetime64[us]
 5   genre                 138 non-null    str           
 6   age                   138 non-null    str           
 7   typeAccompagnement    138 non-null    str           
 8   langue                138 non-null    str           
 9   anonymat              138 non-null    str           
 10  annee                 138 non-null    int32         
 11  mois                  138 non-null    int32         
 12  statut_age            138 non-null    str           
 13  accomp_juridique      138 non-n

In [19]:
# Contrôle systématique de toutes les colonnes catégorielles (texte)
for col in df.select_dtypes(include=["object", "string"]):
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

--- cyberharcelementType ---
<StringArray>
[       'Publication de photos intimes ou personnelles',
                                               'Autres',
                                      'Propos de haine',
 'Menace de publier des photos intimes ou personnelles',
                                          'Diffamation',
                    'Propos raciste ou discriminatoire',
                                'Usurpation d’identité',
                                          'Escroquerie']
Length: 8, dtype: str

--- plateforme ---
<StringArray>
['Facebook', 'Instagram', 'WhatsApp', 'Tiktok']
Length: 4, dtype: str

--- accompagnement ---
<StringArray>
['Non', 'Oui']
Length: 2, dtype: str

--- genre ---
<StringArray>
['Masculin', 'Féminin', 'Non renseigné']
Length: 3, dtype: str

--- age ---
<StringArray>
['+26 ans', '18-25 ans', 'Non renseigné', '13-17 ans']
Length: 4, dtype: str

--- typeAccompagnement ---
<StringArray>
[                    'Suppression', 'Juridique;Psychique;Suppr

## 4. Calcul des KPI

Rappel de la liste validée (8 KPI, KPI 6 « titulaire » retiré) :

1. Évolution du volume de signalements
2. Répartition par genre
3. Répartition par tranche d'âge
4. Typologie des cyberviolences
5. Répartition par plateforme
6. Taux et type d'accompagnement sollicité
7. Taux d'anonymat
8. Répartition par langue



### KPI 1 — Évolution mensuelle du nombre de signalements dans l'échantillon 2025

On complète les 12 mois de l'année (même ceux sans signalement, mis à 0), pour que la
variation mensuelle soit correcte et ne saute pas de mois manquants.

In [20]:
# On force la présence des 12 mois de 2025, même quand il n'y a aucun signalement
tous_les_mois = pd.DataFrame({"mois": range(1, 13), "annee": 2025})

volume_par_mois = df.groupby(["annee", "mois"]).size().reset_index(name="nb_signalements")

volume_mois = tous_les_mois.merge(volume_par_mois, on=["annee", "mois"], how="left")
volume_mois["nb_signalements"] = volume_mois["nb_signalements"].fillna(0).astype(int)
volume_mois["periode"] = volume_mois["annee"].astype(str) + "-" + volume_mois["mois"].astype(str).str.zfill(2)

# Variation % : on évite les "inf" trompeurs quand le mois précédent = 0
volume_mois["variation_%"] = volume_mois["nb_signalements"].pct_change() * 100
volume_mois.loc[volume_mois["nb_signalements"].shift(1) == 0, "variation_%"] = pd.NA
volume_mois["variation_%"] = volume_mois["variation_%"].round(1)

volume_mois

,mois,annee,nb_signalements,periode,variation_%
0,1,2025,32,2025-01,NaN
1,2,2025,31,2025-02,-3.1
2,3,2025,9,2025-03,-71.0
3,4,2025,2,2025-04,-77.8
4,5,2025,0,2025-05,-100.0
5,6,2025,0,2025-06,NaN
6,7,2025,0,2025-07,NaN
7,8,2025,0,2025-08,NaN
8,9,2025,0,2025-09,NaN
9,10,2025,0,2025-10,NaN


In [ ]:
fig1 = px.line(volume_mois, x="periode", y="nb_signalements", markers=True,
               title="KPI 1 — Évolution mensuelle du nombre de signalements dans l'échantillon 2025")
fig1.update_layout(xaxis_title="Mois", yaxis_title="Nombre de signalements")
fig1.show()

### KPI 2 — Répartition par genre

La catégorie "Non renseigné" est affichée dans le graphique, par souci de transparence
(on ne la masque pas).

In [57]:
genre_pct = df["genre"].value_counts(normalize=True).mul(100).round(1)
print(genre_pct)

fig2 = px.pie(df, names="genre", title="KPI 2 — Répartition par genre (échantillon 2025)",color_discrete_sequence=px.colors.qualitative.Pastel)
fig2.show()

genre
Féminin          57.2
Masculin         39.1
Non renseigné     3.6
Name: proportion, dtype: float64


### KPI 3 — Répartition par tranche d'âge

In [58]:
age_pct = df["age"].value_counts(normalize=True).mul(100).round(1)
print(age_pct)

print()
print(df["statut_age"].value_counts(normalize=True).mul(100).round(1))


# KPI 3 — Répartition par âge
fig3 = px.bar(
    df["age"].value_counts().reset_index(),
    x="age",
    y="count",
    color="age",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="KPI 3 — Répartition par tranche d'âge (échantillon 2025)"
)

fig3.update_layout(
    xaxis_title="Tranche d'âge",
    yaxis_title="Nombre de signalements",
    showlegend=False
)

fig3.show()


# KPI 3 bis — Répartition par statut d'âge
fig3b = px.pie(
    df,
    names="statut_age",
    title="KPI 3 (bis) — Répartition Mineur / Jeune adulte / Adulte",
    color="statut_age",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig3b.show()

age
18-25 ans        50.0
+26 ans          42.8
Non renseigné     6.5
13-17 ans         0.7
Name: proportion, dtype: float64

statut_age
Jeune adulte     50.0
Adulte           42.8
Non renseigné     6.5
Mineur            0.7
Name: proportion, dtype: float64


### KPI 4 — Typologie des cyberviolences signalées

In [56]:
type_pct = df["cyberharcelementType"].value_counts(normalize=True).mul(100).round(1)
print(type_pct)

fig4 = px.bar(
    df["cyberharcelementType"].value_counts().reset_index(),
    x="count",
    y="cyberharcelementType",
    orientation="h",
    color="cyberharcelementType",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="KPI 4 — Typologie des cyberviolences signalées (échantillon 2025)"
)

fig4.update_layout(
    xaxis_title="Nombre de signalements",
    yaxis_title="",
    showlegend=False
)

fig4.update_yaxes(categoryorder="total ascending")

fig4.show()

cyberharcelementType
Autres                                                  37.0
Publication de photos intimes ou personnelles           23.9
Propos de haine                                         15.2
Menace de publier des photos intimes ou personnelles     9.4
Diffamation                                              9.4
Propos raciste ou discriminatoire                        3.6
Usurpation d’identité                                    0.7
Escroquerie                                              0.7
Name: proportion, dtype: float64


### KPI 5 — Répartition par plateforme

Le dataset ne contient que 4 plateformes (Facebook, Instagram, WhatsApp, TikTok).
"Messenger" existe comme option dans le formulaire mais  pas dans cet
échantillon — Je l'ajoute pas .

In [55]:
fig5 = px.bar(
    df["plateforme"].value_counts().reset_index(),
    x="plateforme",
    y="count",
    color="plateforme",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="KPI 5 — Répartition par plateforme (échantillon 2025)"
)

fig5.update_layout(
    xaxis_title="Plateforme",
    yaxis_title="Nombre de signalements",
    showlegend=False
)

fig5.show()

### KPI 6 — Taux et type d'accompagnement sollicité

On distingue bien deux choses différentes :
- `accompagnement` (Oui/Non) : la victime a-t-elle demandé un accompagnement (juridique
  et/ou psychologique), en plus de la suppression du contenu ?
- `typeAccompagnement` : le détail de ce qui a été demandé.

La colonne `typeAccompagnement` contient presque toujours "Suppression" (même quand
`accompagnement` = "Non"), car la suppression du contenu est l'action de base d'EMC
Helpline, indépendante de la demande d'accompagnement juridique/psychologique.

In [46]:
accomp_pct = df["accompagnement"].value_counts(normalize=True).mul(100).round(1)
print("Taux d'accompagnement (Oui/Non) :")
print(accomp_pct)
print()
print("Détail par type (%, sur l'ensemble des signalements) :")
for t in types_possibles:
    print(f"  {t} : {df[f'accomp_{t.lower()}'].mean() * 100:.1f} %")

fig6 = px.pie(df, names="accompagnement", title="KPI 6 — Taux d'accompagnement sollicité", color_discrete_sequence=px.colors.qualitative.Set1_r)
fig6.show()

Taux d'accompagnement (Oui/Non) :
accompagnement
Non    76.8
Oui    23.2
Name: proportion, dtype: float64

Détail par type (%, sur l'ensemble des signalements) :
  Juridique : 19.6 %
  Psychique : 10.9 %
  Suppression : 97.8 %


In [65]:
accompagnement = pd.DataFrame({
    "Type d'accompagnement": ["Juridique", "Psychique", "Suppression"],
    "Pourcentage": [19.6, 10.9, 97.8]
})

fig6b = px.bar(
    accompagnement,
    x="Type d'accompagnement",
    y="Pourcentage",
    color="Type d'accompagnement",
    text="Pourcentage",  # <-- on passe le nom de la colonne ici, pas un tableau à part
    labels={
        "Type d'accompagnement": "Type d'accompagnement",
        "Pourcentage": "Pourcentage (%)"
    },
    title="KPI 6 (détail) — Type d'accompagnement sollicité",
    color_discrete_sequence=px.colors.qualitative.Set1_r
)
fig6b.update_traces(texttemplate="%{text}%", textposition="outside")
fig6b.show()

### KPI 7 — Taux d'anonymat

Calcul : nombre de "Oui" / nombre total × 100.

In [40]:
anonymat_pct = df["anonymat"].value_counts(normalize=True).mul(100).round(1)
print(anonymat_pct)

fig7 = px.pie(df, names="anonymat", title="KPI 7 — Taux d'anonymat", color_discrete_sequence=px.colors.qualitative.Set2)
fig7.show()

anonymat
Oui    76.1
Non    23.9
Name: proportion, dtype: float64


### KPI 8 — Répartition par langue

In [37]:
langue_pct = df["langue"].value_counts(normalize=True).mul(100).round(1)
print(langue_pct)

fig8 = px.pie(df, names="langue", title="KPI 8 — Répartition par langue",color_discrete_sequence=px.colors.qualitative.Set3)
fig8.show()

langue
fr    62.3
ar    37.7
Name: proportion, dtype: float64


## 5. Contrôles qualité finaux

Avant d'exporter le dataset nettoyé, on vérifie systématiquement qu'aucune donnée n'a
été perdue ou corrompue pendant le nettoyage.

In [29]:
# 1. Doublons d'ID
print("Nombre de doublons d'ID :", df["id"].duplicated().sum())

Nombre de doublons d'ID : 0


In [30]:
# 2. Valeurs manquantes après nettoyage (doit être 0 partout, sauf colonnes non concernées)
df.isna().sum()

id                      0
cyberharcelementType    0
plateforme              0
accompagnement          0
date                    0
genre                   0
age                     0
typeAccompagnement      0
langue                  0
anonymat                0
annee                   0
mois                    0
statut_age              0
accomp_juridique        0
accomp_psychique        0
accomp_suppression      0
dtype: int64

In [31]:
# 3. Nombre final de lignes : doit toujours être 138 (aucune ligne supprimée)
print("Nombre de lignes :", df.shape[0])
assert df.shape[0] == 138, "Attention : le nombre de lignes a changé pendant le nettoyage !"
print("OK : aucune ligne n'a été supprimée.")

Nombre de lignes : 138
OK : aucune ligne n'a été supprimée.


In [32]:
# 4. Vérification des dates
print("Date min :", df["date"].min())
print("Date max :", df["date"].max())

Date min : 2025-01-01 00:00:00
Date max : 2025-12-17 00:00:00


In [33]:
# 5. Contrôle final de toutes les variables catégorielles
for col in df.select_dtypes(include=["object", "string"]):
    print(f"--- {col} ---")
    print(sorted(df[col].dropna().unique()))
    print()

--- cyberharcelementType ---
['Autres', 'Diffamation', 'Escroquerie', 'Menace de publier des photos intimes ou personnelles', 'Propos de haine', 'Propos raciste ou discriminatoire', 'Publication de photos intimes ou personnelles', 'Usurpation d’identité']

--- plateforme ---
['Facebook', 'Instagram', 'Tiktok', 'WhatsApp']

--- accompagnement ---
['Non', 'Oui']

--- genre ---
['Féminin', 'Masculin', 'Non renseigné']

--- age ---
['+26 ans', '13-17 ans', '18-25 ans', 'Non renseigné']

--- typeAccompagnement ---
['Juridique', 'Juridique;Psychique;Suppression', 'Juridique;Suppression', 'Psychique;Suppression', 'Suppression']

--- langue ---
['ar', 'fr']

--- anonymat ---
['Non', 'Oui']

--- statut_age ---
['Adulte', 'Jeune adulte', 'Mineur', 'Non renseigné']



## 6. Vérification manuelle des calculs

On recompte ici 2 KPI à la main sur un sous-échantillon.

In [ ]:
# Vérif manuelle KPI 2 (genre) sur les 20 premières lignes
echantillon = df.head(20)
print(echantillon["genre"].value_counts())
print("Total échantillon :", len(echantillon))


genre
Féminin     12
Masculin     8
Name: count, dtype: int64
Total échantillon : 20


In [35]:
# Vérif manuelle KPI 5 (plateforme) sur les 20 premières lignes
print(echantillon["plateforme"].value_counts())

plateforme
Instagram    14
Facebook      6
Name: count, dtype: int64


#### BILAN:
##### J’ai vérifié manuellement ces résultats, et ils sont cohérents.

## 7. Sauvegarde du dataset nettoyé

J'ai exporté le dataset nettoyé pour le réutiliser directement dans l'application
Streamlit du Jalon 3 .

`encoding="utf-8-sig"` évite les problèmes d'accents à l'ouverture du CSV dans Excel
sous Windows.

In [36]:
df.to_csv("signalements_clean.csv", index=False, encoding="utf-8-sig")
print("Fichier exporté : signalements_clean.csv")
df.head()

Fichier exporté : signalements_clean.csv


,id,cyberharcelementType,plateforme,accompagnement,date,genre,age,typeAccompagnement,langue,anonymat,annee,mois,statut_age,accomp_juridique,accomp_psychique,accomp_suppression
0,979,Publication de photos intimes ou personnelles,Facebook,Non,2025-01-01,Masculin,+26 ans,Suppression,fr,Oui,2025,1,Adulte,0,0,1
1,980,Publication de photos intimes ou personnelles,Instagram,Non,2025-01-02,Féminin,18-25 ans,Suppression,fr,Oui,2025,1,Jeune adulte,0,0,1
2,981,Publication de photos intimes ou personnelles,Facebook,Oui,2025-01-03,Féminin,18-25 ans,Juridique;Psychique;Suppression,ar,Non,2025,1,Jeune adulte,1,1,1
3,982,Autres,Facebook,Non,2025-01-04,Féminin,+26 ans,Suppression,fr,Oui,2025,1,Adulte,0,0,1
4,983,Autres,Facebook,Non,2025-01-05,Féminin,18-25 ans,Suppression,fr,Oui,2025,1,Jeune adulte,0,0,1


## 8. Limites des données

Points à garder en tête pour l'interprétation de tous les KPI ci-dessus, et à mentionner
dans le rapport de stage :

- Le dataset utilisé est un **échantillon transmis par l'encadrante** (138 lignes), pas
  l'intégralité des signalements EMC Helpline 2025.
- Certaines informations sont **manquantes** (`genre` : 5 valeurs, `age` : 9 valeurs),
  regroupées sous "Non renseigné" plutôt que supprimées ou estimées.
- Les **identifiants (`id`) présentent des discontinuités** (ex. 1042 → 1351, 1414 → 1668),
  ce qui confirme qu'il s'agit d'un extrait partiel plutôt que d'une séquence continue.
- Certaines catégories prévues par le formulaire de signalement en ligne (ex. tranche
  "5-12 ans", plateforme "Messenger") **n'apparaissent pas** dans cet échantillon — elles
  ne sont donc pas représentées dans les graphiques, sans que cela signifie qu'elles
  n'existent pas dans la réalité des signalements EMC Helpline.
- **Conclusion :** les résultats des KPI décrivent l'échantillon disponible, et non
  nécessairement l'ensemble des signalements EMC Helpline 2025.